<a href="https://colab.research.google.com/github/Schlomo-khalidi/PytorchML/blob/main/Creating%26Training_ML_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import torch.nn as nn

In [9]:
#Donloading the training Datasets
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

#Downloading the test Dataset
test_data = datasets.FashionMNIST(
        root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)


100%|██████████| 26.4M/26.4M [00:01<00:00, 22.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 340kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 6.24MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.1MB/s]


In [12]:
bach_size = 64
#Create data Loaders
train_dataloader = DataLoader(training_data, batch_size=bach_size)
test_dataloader = DataLoader(test_data, batch_size=bach_size)

for X, y in test_dataloader:
  print(f"Shape of X [N,C,H,W]: {X.shape}")
  print(f"Shape of y: {y.shape} {y.dtype}")
  break

Shape of X [N,C,H,W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


# ***Creating The Model***

In [19]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"using {device} device")

#Define Class
class NeuralNetwork(nn.Module):
    def __init__(self):
      super().__init__()
      self.flatten = nn.Flatten()
      self.linear_relu_stack = nn.Sequential(
          nn.Linear(28*28, 512),
          nn.ReLU(),
          nn.Linear(512, 512),
          nn.ReLU(),
          nn.Linear(512, 10),
      )
    def forward(self, x):
      x = self.flatten(x)
      logits = self.linear_relu_stack(x)
      return logits
model = NeuralNetwork().to(device)
print(model)



using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


# ***Optimizing Model Parameters***

In [20]:
loss_fn= nn.CrossEntropyLoss()
optimizer= torch.optim.SGD(model.parameters(), lr=1e-3)

In [21]:
def train(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  model.train()
  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    #compute prediction error
    perd = model(X)
    loss = loss_fn(perd, y)

    #Backpropagation
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), batch * len(X)
      print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")


In [22]:
def test(dataloader, model, loss_fn):
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  model.eval()
  test_loss, correct = 0, 0
  with torch.no_grad():
    for X, y in dataloader:
      X, y = X.to(device), y.to(device)
      pred = model(X)
      test_loss += loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()
  test_loss /= num_batches
  correct /= size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [23]:
epochs = 5
for t in range(epochs):
  print(f"Epoch {t+1}\n-------------------------------")
  train(train_dataloader, model, loss_fn, optimizer)
  test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.307346 [    0/60000]
loss: 2.292843 [ 6400/60000]
loss: 2.276586 [12800/60000]
loss: 2.267568 [19200/60000]
loss: 2.243198 [25600/60000]
loss: 2.214744 [32000/60000]
loss: 2.229245 [38400/60000]
loss: 2.189168 [44800/60000]
loss: 2.179167 [51200/60000]
loss: 2.148552 [57600/60000]
Test Error: 
 Accuracy: 41.5%, Avg loss: 2.145060 

Epoch 2
-------------------------------
loss: 2.152617 [    0/60000]
loss: 2.141903 [ 6400/60000]
loss: 2.083383 [12800/60000]
loss: 2.098761 [19200/60000]
loss: 2.044873 [25600/60000]
loss: 1.982942 [32000/60000]
loss: 2.010708 [38400/60000]
loss: 1.924934 [44800/60000]
loss: 1.922795 [51200/60000]
loss: 1.846680 [57600/60000]
Test Error: 
 Accuracy: 57.8%, Avg loss: 1.850854 

Epoch 3
-------------------------------
loss: 1.886343 [    0/60000]
loss: 1.852539 [ 6400/60000]
loss: 1.735722 [12800/60000]
loss: 1.771621 [19200/60000]
loss: 1.672184 [25600/60000]
loss: 1.625851 [32000/60000]
loss: 1.644264 [38400/

# ***Saving My Model***

In [24]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


# ***Loading My Model***

In [26]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

***Testing if this model can make predictions 👌👌💪***

In [27]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
